# Exploring Consumer Finance Data

# Introduction

For the first five projects of this course we always had a **target to
predict** — a price, a category, a probability of bankruptcy. We are now
crossing into **unsupervised learning**, where the data arrives **without
labels** and the goal shifts:

> ❓ If no one tells us which households are "similar," can we let the
> data reveal its own groupings?

That is the promise of **clustering**, and over the next four notebooks
we will use it to **segment U.S. consumers** by their financial
profiles. But before any algorithm can find structure, *we* have to
understand the raw material. This first notebook is pure **exploratory
data analysis (EDA)** of the **Survey of Consumer Finances (SCF)**: we
examine distributions, compare subgroups, and measure relationships so
that — by L2 — we know which features are worth clustering on.

By the end of this notebook you will be able to:

-   Load and inspect a large consumer finance dataset using pandas.
-   Subset data based on categorical conditions.
-   Create bar charts, histograms, and scatter plots to explore
    distributions.
-   Compute and interpret correlation matrices.
-   Identify meaningful differences between consumer subgroups.

In [ ]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1170268017", h="3298dbabb7", width=700, height=450) 

# 1. Conceptual Foundation

## Project context: what we're trying to build

Before we touch a clustering algorithm, we need to be fluent in the
**dataset** and in the **pandas moves** that EDA relies on. This section
builds that fluency on a tiny toy DataFrame, then the *Applied Exercises*
repeat every move on the real SCF data.

> 💡 Why explore before clustering?
>
> - Clustering groups points by **distance**, so features on wild
>   scales or with heavy outliers can hijack the result.
> - We want to enter L2 already knowing **which variables vary, which
>   move together, and where the subgroups differ.**
> - EDA is also where we catch encoding quirks (numeric codes standing
>   in for categories) that would otherwise corrupt the analysis.

The [Survey of Consumer Finances](https://www.federalreserve.gov/econres/scfindex.htm)
(SCF) is a triennial survey sponsored by the U.S. Federal Reserve. It
collects detailed information about American households' balance sheets,
pensions, income, and demographic characteristics, and is widely used by
economists and policymakers to understand financial behavior in the
United States.

Our **guiding question** for the EDA:

> **What financial and demographic characteristics distinguish
> households that fear being denied credit from those that do not?**

This question gives every chart a purpose — we are always comparing the
**credit-fearful** subgroup against everyone else.

## Understanding the dataset structure

The SCF extract we use is the **2019** wave: large and wide — over
**20,000 rows** and **350+ columns**.

> 🔍 What each part of the table means
>
> - **Rows** = households (the unit of observation).
> - **Columns** = financial variables (assets, debts, income),
>   demographic variables (age, race, education), and
>   behavioral/attitudinal variables.
> - **Many categories are encoded as integers** (e.g., age groups as
>   1–6) — a storage convention we must decode before plotting.
> - **Monetary values are inflation-adjusted to 2019 dollars.**

The `TURNFEAR` column is our subgroup switch: it flags households that
were **turned down for credit or feared being denied** it in the past 5
years (`1` = yes, `0` = no).

Let's rehearse the core inspection methods on a small toy DataFrame
before meeting the real thing:

In [ ]:
import pandas as pd

# Create a small toy DataFrame to demonstrate inspection methods
toy_data = {
    "household_id": [1, 2, 3, 4, 5],
    "income": [50000, 75000, 120000, 45000, 90000],
    "debt": [10000, 25000, 50000, 5000, 30000],
    "age_group": [2, 3, 4, 1, 3],
    "credit_fear": [1, 0, 0, 1, 0],
}
toy_df = pd.DataFrame(toy_data)

# Display the shape (rows, columns)
print("Shape:", toy_df.shape)

# Display the first few rows
toy_df.head()

In [ ]:
# Check data types for each column
print(toy_df.dtypes)

In [ ]:
# Get summary statistics for numeric columns
toy_df.describe()

> 📊 Reading the inspection output
>
> - `.shape` confirms the table's size as `(rows, columns)` — always the
>   first sanity check after loading.
> - `.dtypes` tells you how pandas *typed* each column; here every column
>   is numeric, but on real survey data a column can silently load as
>   `object` (text) and break arithmetic.
> - `.describe()` summarizes the numeric spread (count, mean, std, min,
>   quartiles, max) — your first look at scale and outliers.

> 📌 Key methods
>
> - [`read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html)
>   reads compressed files (`.csv.gz`) directly by inferring compression
>   from the extension.
> - [`DataFrame.shape`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.shape.html)
>   returns `(rows, columns)`;
>   [`head`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html)
>   shows the first n rows;
>   [`dtypes`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dtypes.html)
>   shows per-column types;
>   [`describe`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html)
>   summarizes numeric columns.

➡️ Next we isolate a **subgroup** — the move that powers every "fearful
vs. everyone" comparison in this notebook.

## Subsetting data with boolean masks

A **boolean mask** is a Series of `True`/`False` values, one per row.
Indexing a DataFrame with that mask keeps only the `True` rows.

> 🧠 The three-step intuition
>
> 1. Write a **condition** (`df["credit_fear"] == 1`) → pandas evaluates
>    it row-by-row into a `True`/`False` Series.
> 2. **Pass the mask** to `df[mask]` → only `True` rows survive.
> 3. **Combine** conditions with `&` (and) / `|` (or), wrapping each in
>    parentheses: `(a) & (b)`.

In [ ]:
# Create a boolean mask: True where credit_fear equals 1
mask = toy_df["credit_fear"] == 1
print("Boolean mask:")
print(mask)

In [ ]:
# Apply the mask to filter rows
subset = toy_df[mask]
print(f"Original rows: {len(toy_df)}, Subset rows: {len(subset)}")
subset

> 📊 What just happened
>
> The mask printed as a `True`/`False` Series aligned to the row index;
> filtering with it shrank the toy frame from 5 rows to just the
> credit-fearful ones. On the real data this is exactly how we will carve
> out `df_fear`.

> ⚠️ Common pitfall — mask length
>
> The mask must have the **same length and index** as the DataFrame you
> apply it to. Building a mask from one DataFrame and applying it to
> another (or to a filtered copy) gives misaligned or empty results.

➡️ Real surveys store categories as **numbers**, so before we can chart
"age groups" we must translate codes into labels.

## Working with categorical data encoded as numbers

Survey data often encodes categorical variables as integers to save
storage. For example, an age-group column might use:

| Code | Meaning  |
|------|----------|
| 1    | Under 35 |
| 2    | 35-44    |
| 3    | 45-54    |
| 4    | 55-64    |

Before visualizing, we replace these codes with human-readable labels
using [`Series.replace`](https://pandas.pydata.org/docs/reference/api/pandas.Series.replace.html)
and a dictionary mapping.

In [ ]:
# Dictionary mapping numeric codes to labels
age_dict = {1: "Under 35", 2: "35-44", 3: "45-54", 4: "55-64"}

# Replace codes with labels
age_labels = toy_df["age_group"].replace(age_dict)
print("Original codes:")
print(toy_df["age_group"].values)
print("\nAfter replacement:")
print(age_labels.values)

In [ ]:
# Check unique values before and after replacement
print("Unique codes:", toy_df["age_group"].unique())
print("Unique labels:", age_labels.unique())

> 📊 Before and after
>
> The same Series prints first as bare integers, then as descriptive
> strings — and `.unique()` confirms the set of distinct values changed
> from codes to labels. Decoding like this is what makes a bar chart
> readable instead of a row of mystery numbers.

> ⚠️ Always check the Code Book first
>
> Never guess what a numeric code means. For the SCF, consult the
> official [Code Book](https://sda.berkeley.edu/sdaweb/docs/scfcomb2019/DOC/hcbk.htm).
> A wrong mapping silently mislabels every downstream chart.

> 📌 Key methods
>
> - [`replace`](https://pandas.pydata.org/docs/reference/api/pandas.Series.replace.html)
>   maps old values to new via a dictionary.
> - [`unique`](https://pandas.pydata.org/docs/reference/api/pandas.Series.unique.html)
>   lists distinct values;
>   [`value_counts`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html)
>   counts how often each appears.

➡️ With labels in hand, the natural next question is *how many* fall in
each category — and how to compare that fairly across groups.

## Counting and normalizing frequencies

When exploring a categorical variable, `value_counts()` is the workhorse:
it counts how many times each unique value appears.

In [ ]:
# Count occurrences of each age group label
age_labels.value_counts()

> ❗️ But raw counts can mislead
>
> When comparing groups of **different sizes**, raw counts deceive. If
> Group A has 1,000 people and Group B has 100, "200 from A vs. 50 from
> B" tells you little about *proportions* — B is actually far more
> concentrated in that category.
>
> **Normalized frequencies** fix this by dividing each count by the group
> total, so every group sums to 1 and becomes directly comparable:

In [ ]:
# Normalized frequencies sum to 1.0
age_labels.value_counts(normalize=True)

> ✅ Sanity check
>
> Normalized frequencies should always sum to **≈ 1.0**. If they don't,
> something was filtered or grouped incorrectly.

> 📌 Key options
>
> - `value_counts()` returns counts in descending order by default.
> - `normalize=True` returns **proportions** instead of raw counts.
> - `sort_index=True` orders by label/index instead of by count.

➡️ Proportions within one column are useful; comparing them **across a
second column** is where `groupby` comes in.

## Grouped aggregations with groupby

To compare statistics **across subgroups**, pandas uses the
**split-apply-combine** pattern via
[`groupby`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html):
split the data into groups, apply a function to each, and combine the
results.

> 🧠 Split → apply → combine
>
> - **Split** the rows by the values of a key column (`credit_fear`).
> - **Apply** an aggregation (`mean`, `value_counts`) within each group.
> - **Combine** the per-group answers into one tidy result.

In [ ]:
# Group by credit_fear and compute mean income for each group
toy_df.groupby("credit_fear")["income"].mean()

In [ ]:
# Combine groupby with value_counts for cross-tabulation
# Count age groups within each credit_fear category
(
    toy_df["age_group"]
    .replace(age_dict)
    .groupby(toy_df["credit_fear"])
    .value_counts(normalize=True)
)

> 📊 Reading the grouped result
>
> The first cell gives one mean income **per** `credit_fear` value — a
> direct fearful-vs-not comparison in a single line. The second chains
> `groupby` with `value_counts(normalize=True)` to get a
> **cross-tabulation**: the age-group mix *within* each fear status.

> 📌 Key moves
>
> - `groupby` splits data by column values.
> - Chain `.value_counts()` after it to count within each group.
> - `.reset_index()` turns the grouped result back into a flat DataFrame
>   that plots cleanly.

➡️ Numbers in a table are hard to compare at a glance — time to
**visualize**.

## Creating visualizations with pandas and matplotlib

Pandas integrates with matplotlib so you can plot **directly** off a
Series or DataFrame. Three chart types cover most EDA:

> ✅ Which chart for which question?
>
> - **Bar chart** (`kind="bar"`) — compare counts across categories.
> - **Histogram** (`.hist`) — see the shape of one numeric variable.
> - **Scatter** (`.plot.scatter`) — reveal the relationship between two
>   numeric variables.

In [ ]:
import matplotlib.pyplot as plt

# Bar chart from value counts
counts = age_labels.value_counts()
counts.plot(kind="bar", xlabel="Age Group", ylabel="Count",
            title="Age Group Distribution")
plt.tight_layout();

In [ ]:
# Histogram of a numeric column
toy_df["income"].hist(bins=5, edgecolor="black")
plt.xlabel("Income")
plt.ylabel("Frequency")
plt.title("Income Distribution");

In [ ]:
# Scatter plot to visualize relationships
toy_df.plot.scatter(x="income", y="debt")
plt.title("Income vs Debt");

> 📊 What each plot shows
>
> - The **bar chart** ranks age groups by count — the tallest bar is the
>   most common group.
> - The **histogram** bins income into ranges; its shape (right-skewed,
>   bimodal, …) is what a single number like the mean hides.
> - The **scatter** of income vs. debt shows whether the two rise
>   together — a visual preview of correlation.

> 🔧 Debugging tip
>
> If a plot looks empty or wrong, check that the column holds valid
> numbers (not all `NaN`), the column name is spelled correctly, and the
> plot type matches the data.

➡️ To put **two groups side by side** in one chart, we reach for seaborn.

## Creating comparative visualizations with seaborn

Seaborn extends matplotlib with statistical plots built for **group
comparison**. Its `hue` parameter is the key: it splits a chart by a
categorical variable into side-by-side bars.

In [ ]:
import seaborn as sns

# Prepare data for seaborn: need a tidy DataFrame
# Compute frequencies by credit_fear and age_group
df_freq = (
    toy_df["age_group"]
    .replace(age_dict)
    .groupby(toy_df["credit_fear"])
    .value_counts(normalize=True)
    .rename("frequency")
    .reset_index()
)
df_freq

In [ ]:
# Side-by-side bar chart comparing groups
sns.barplot(x="age_group", y="frequency", hue="credit_fear", data=df_freq)
plt.xlabel("Age Group")
plt.ylabel("Frequency")
plt.title("Age Distribution by Credit Fear Status");

> 📊 Reading the grouped bars
>
> Each x-position shows two bars — one per `credit_fear` value — so you
> can read off *where the groups diverge* at a glance. That side-by-side
> contrast is exactly the comparison our guiding question asks for.

> 📌 Key points
>
> - [`barplot`](https://seaborn.pydata.org/generated/seaborn.barplot.html)
>   draws grouped bars via `hue`.
> - Seaborn expects **tidy data**: one row per observation, with columns
>   for x, y, and the grouping variable (hence the `reset_index` earlier).
> - `order=` controls the category order on the x-axis.

➡️ Bars compare *distributions*; to quantify how two variables **move
together**, we turn to correlation.

## Correlation analysis

Correlation measures the **linear** relationship between two numeric
variables on a fixed scale:

> 🧠 The −1 … 0 … +1 scale
>
> - **+1** — perfect positive: as X rises, Y rises proportionally.
> - **0** — no *linear* relationship.
> - **−1** — perfect negative: as X rises, Y falls proportionally.

A single `Series.corr` gives one pair's coefficient;
`DataFrame.corr` gives the full matrix across many columns, which we can
color with a gradient to spot strong relationships instantly.

In [ ]:
# Compute correlation between two columns
corr_value = toy_df["income"].corr(toy_df["debt"])
print(f"Correlation between income and debt: {corr_value:.3f}")

In [ ]:
# Compute correlation matrix for all numeric columns
corr_matrix = toy_df[["income", "debt"]].corr()
corr_matrix

In [ ]:
# Style the correlation matrix with color gradients
corr_matrix.style.background_gradient(axis=None, cmap="coolwarm")

> 📊 Reading the styled matrix
>
> The diagonal is always `1.0` (a variable correlates perfectly with
> itself). Off-diagonal cells, shaded by the `coolwarm` gradient, show
> strength and sign: deep warm = strong positive, deep cool = strong
> negative, pale = weak. We will read the **real** SCF matrix this way in
> §7 — and again for the credit-fearful subset, to see whether
> relationships *differ* by group (a clue for which features to cluster
> on).

> ⚠️ Correlation ≠ causation
>
> A high correlation between debt and house value does **not** mean one
> causes the other. Correlation also captures only **linear** patterns.

> 🔧 If a correlation looks surprising, check for
>
> - **Outliers** distorting the relationship.
> - **Non-linear** structure that correlation can't see.
> - **Missing values** shrinking the effective sample.

> 📌 Key methods
>
> - [`Series.corr`](https://pandas.pydata.org/docs/reference/api/pandas.Series.corr.html)
>   for one pair;
>   [`DataFrame.corr`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html)
>   for the full matrix;
>   [`background_gradient`](https://pandas.pydata.org/docs/reference/api/pandas.io.formats.style.Styler.background_gradient.html)
>   to color it.

➡️ The toy rehearsal is over. Time to run every one of these moves on the
**real SCF data**.

# Applied Exercises

## 2. Setup

We begin by importing the libraries this notebook needs. It is good
practice to keep all imports in a single cell at the top so the
environment is reproducible.

**Code 6.1.2.1**:

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

## 3. Load and Inspect the Data

### Problem

Before we can explore consumer financial behavior, we need to load the
SCF dataset and understand its basic structure. The data is stored in a
compressed CSV file, and we need to identify the subset of households
that fear credit denial for our comparative analysis.

### Approach

We will load the compressed CSV with `pandas.read_csv` (which handles
`.gz` automatically), then build the `df_fear` subset where
`TURNFEAR == 1` — the focus of much of the analysis.

### Tasks

Load the SCF dataset from the compressed CSV file. You should see a
DataFrame with over 20,000 rows and 351 columns.

**Code Task 6.1.3.1**:

In [ ]:
# Load the SCF dataset from compressed CSV
df = pd.read_csv('data/SCFP2019.csv.gz')
print('df shape:', df.shape)
df.head()


Now create a boolean mask for households where `TURNFEAR == 1` and apply
it. The resulting `df_fear` should be **smaller** than the original `df`.

**Code Task 6.1.3.2**:

In [ ]:
# Create a mask for credit-fearful households (TURNFEAR == 1)
mask = df['TURNFEAR'] == 1  # condition for TURNFEAR equals 1
df_fear = df[mask]
print("df_fear shape:", df_fear.shape)
df_fear.head()

### Checkpoint

> 🧪 What this verifies
>
> The asserts confirm the load and subset are correct **before** we build
> analysis on top of them: 351 columns, more than 20,000 rows, `df_fear`
> strictly smaller than `df`, and every row of `df_fear` truly has
> `TURNFEAR == 1`.

In [ ]:
# Verify dataset dimensions and subset
assert df.shape[1] == 351, (
    f"Expected 351 columns in df, got {df.shape[1]}"
)
assert df.shape[0] > 20000, (
    f"Expected more than 20000 rows in df, got {df.shape[0]}"
)
assert df_fear.shape[0] < df.shape[0], (
    "df_fear should be a subset smaller than df"
)
assert (df_fear["TURNFEAR"] == 1).all(), (
    "All rows in df_fear should have TURNFEAR == 1"
)
print("All checks passed!")

> ✅ Checks passed
>
> The dataset loaded at its expected dimensions and the subgroup is
> clean. Everything downstream can trust `df` and `df_fear`.

➡️ With the data trustworthy, our first comparison is **age** — are
younger households more credit-fearful?

## 4. Explore Age Distribution

### Problem

Understanding the age distribution of credit-fearful households helps
identify which demographic groups are most concerned about credit
access. Are younger people more worried, or is this concern spread evenly
across age groups?

### Approach

We will examine the `AGECL` (age class) categorical variable first,
replacing numeric codes with descriptive labels, then chart age-group
frequencies. For finer detail, we also histogram the continuous `AGE`
variable.

### Tasks

Extract the unique values from the `AGECL` column to see which numeric
codes are present. You should see integers from 1 to 6.

**Code 6.1.4.1**:

In [ ]:
# Get unique age group codes to understand the encoding
age_groups = df_fear["AGECL"].unique()  # unique values from df_fear["AGECL"]
print("Age Groups:", age_groups)

Use the dictionary mapping to replace numeric codes with human-readable
labels. The resulting Series should contain strings like "Under 35"
instead of integers.

**Code 6.1.4.2**:

In [ ]:
# Dictionary mapping age codes to labels (from Code Book)
agecl_dict = {
    1: "Under 35",
    2: "35-44",
    3: "45-54",
    4: "55-64",
    5: "65-74",
    6: "75 or Older",
}

# Replace numeric codes with descriptive labels
age_cl = df_fear["AGECL"].replace(agecl_dict)
age_cl.head()

Compute value counts for the labeled age groups and create a bar chart.
The chart shows which age groups hold the most credit-fearful households.

**Code 6.1.4.3**:

In [ ]:
# Create bar chart of age group frequencies
age_cl_value_counts = age_cl.value_counts()  # value_counts of age_cl

# Plot bar chart
age_cl_value_counts.plot(
    kind="bar",
    xlabel="Age Group",
    ylabel="Frequency (count)",
    title="Credit Fearful: Age Groups"
);

Create a histogram of the continuous `AGE` variable for a more detailed
distribution that reveals patterns *within* the broad age categories.

**Code 6.1.4.4**:

In [ ]:
# Create histogram of continuous AGE variable for finer detail
df_fear["AGE"].hist(bins=10)  # histogram of df_fear["AGE"] with 10 bins
plt.xlabel("Age")
plt.ylabel("Frequency (count)")
plt.title("Credit Fearful: Age Distribution");

> 📊 What the age charts reveal
>
> The **bar chart** ranks the six age classes by count, so the tallest
> bars name the age brackets where credit fear concentrates. The
> **histogram** of the continuous `AGE` then shows the finer shape inside
> those brackets — for example whether a "Under 35" bar is driven by
> people in their late twenties or spread evenly. Read the two together:
> the bar chart for the headline, the histogram for the nuance.

### Checkpoint

> 🧪 What this verifies
>
> That there are exactly 6 age groups, that `age_cl` now holds string
> labels (`dtype == object`), and that the value counts still sum to the
> subgroup size (no rows lost in recoding).

In [ ]:
# Verify age group processing
assert len(age_groups) == 6, (
    f"Expected 6 unique age groups, got {len(age_groups)}"
)
assert age_cl.dtype == object, (
    "age_cl should contain string labels after replacement"
)
assert age_cl_value_counts.sum() == len(df_fear), (
    f"Value counts should sum to {len(df_fear)}, got {age_cl_value_counts.sum()}"
)
print("All checks passed!")

> ✅ Recoding preserved every row, and the labels are in place.

➡️ Age is one lens; **race** is another — and here the comparison to the
full population matters most.

## 5. Explore Race Distribution

### Problem

Understanding racial composition helps identify whether credit concerns
affect different demographic groups disproportionately. This requires
comparing the credit-fearful subset to the overall population to draw
meaningful conclusions.

### Approach

We will create horizontal bar charts of **normalized** frequencies of
racial groups for both the credit-fearful subset and the full dataset.
Normalizing lets us compare despite very different group sizes.

> ⚠️ A note on categories
>
> The SCF uses specific racial categories that may not represent all
> groups (note that code 4 is absent from this extract). Always consult
> the [data dictionary](./data-dictionary.ipynb) to understand what each
> code represents and its limitations.

### Tasks

Replace numeric race codes with labels and compute normalized
frequencies for the credit-fearful subset. The horizontal bar chart shows
proportions (0 to 1) rather than raw counts.

**Code Task 6.1.5.1**:

In [ ]:
# Dictionary mapping race codes to labels (from Code Book)
# Note: Code 4 does not exist in this dataset
race_dict = {
    1: "White/Non-Hispanic",
    2: "Black/African-American",
    3: "Hispanic",
    5: "Other",
}

# Create normalized bar chart for credit-fearful subset
race = df_fear['RACE'].replace(race_dict)  # replace codes using race_dict
race_value_counts = race.value_counts(normalize=True)  # normalized value_counts

# Plot horizontal bar chart
race_value_counts.plot(kind="barh")
plt.xlim((0, 1))
plt.xlabel("Frequency (%)")
plt.ylabel("Race")
plt.title("Credit Fearful: Racial Groups");

Now create the **same** visualization for the full dataset. Comparing the
two charts reveals whether certain groups are over- or under-represented
among credit-fearful households.

**Code 6.1.5.2**:

In [ ]:
# Compare with full dataset to understand baseline proportions
race_full = df["RACE"].replace(race_dict)  # replace codes in df["RACE"] using race_dict
race_full_counts = race_full.value_counts(normalize=True)  # normalized value_counts

# Plot horizontal bar chart for full dataset
race_full_counts.plot(kind="barh")
plt.xlim((0, 1))
plt.xlabel("Frequency (%)")
plt.ylabel("Race")
plt.title("SCF Respondents: Racial Groups");

> 📊 Comparing the two bar charts
>
> Because both charts use proportions on the same `[0, 1]` x-axis, you
> can lay them side by side and read **differences directly**: a racial
> group whose bar is longer in the credit-fearful chart than in the
> full-population chart is over-represented among households that fear
> credit denial. That gap — not the raw height — is the finding.

### Checkpoint

> 🧪 What this verifies
>
> That both sets of normalized frequencies sum to **≈ 1.0**, confirming
> the proportions are well-formed for comparison.

In [ ]:
# Verify normalized frequencies sum to ~1.0
assert abs(race_value_counts.sum() - 1.0) < 0.01, (
    f"Normalized frequencies should sum to ~1.0, got {race_value_counts.sum()}"
)
assert abs(race_full_counts.sum() - 1.0) < 0.01, (
    f"Full dataset frequencies should sum to ~1.0, got {race_full_counts.sum()}"
)
print("All checks passed!")

> ✅ Both distributions are valid proportions.

➡️ Demographics set the stage; **income** is where financial pressure
becomes concrete.

## 6. Compare Income Distribution Across Groups

### Problem

Income level likely influences credit concerns. We want to compare the
income distribution of credit-fearful households against non-fearful
households to identify patterns.

### Approach

We will use `groupby` to compute normalized frequencies of income
categories (`INCCAT`) **separately** for each `TURNFEAR` group, then draw
a side-by-side seaborn bar chart of the comparison.

### Tasks

Compute normalized frequencies of income categories grouped by
`TURNFEAR`. The method chain replaces codes, groups by fear status,
computes proportions, and resets the index into a tidy DataFrame ready
for seaborn.

**Code Task 6.1.6.1**:

In [ ]:
# Dictionary mapping income percentile codes to labels
inccat_dict = {
    1: "0-20",
    2: "21-39.9",
    3: "40-59.9",
    4: "60-79.9",
    5: "80-89.9",
    6: "90-100",
}

# Compute normalized frequencies by TURNFEAR group
# Chain: replace -> groupby -> value_counts -> rename -> reset_index
df_inccat = (
    df["INCCAT"]
    .replace(inccat_dict)
    .groupby(df["TURNFEAR"])
    .value_counts(normalize=True)
    .rename('frequency')
    .reset_index()
)

df_inccat

Create a side-by-side bar chart comparing income distributions. The `hue`
parameter separates credit-fearful (1) from non-fearful (0) households,
making differences visually obvious.

**Code 6.1.6.2**:

In [ ]:
# Create side-by-side bar chart comparing income distributions
sns.barplot(
    x="INCCAT",  # x=..., y=..., hue=...
    y="frequency",  # data=df_inccat
    hue="TURNFEAR",  # order=inccat_dict.values()
    data=df_inccat,
    order=inccat_dict.values(),
)
plt.xlabel("Income Category")
plt.ylabel("Frequency (%)")
plt.title("Income Distribution: Credit Fearful vs. Non-fearful");

> 📊 Reading the income comparison
>
> Scan the lowest income categories (`0-20`, `21-39.9`): if the
> credit-fearful bars tower over the non-fearful ones there, it says
> credit fear concentrates among **lower-income** households. The high
> categories (`90-100`) usually show the reverse. The `order=` argument
> keeps the categories in their natural low→high sequence so the trend
> reads left to right.

### Checkpoint

> 🧪 What this verifies
>
> That the tidy DataFrame has the expected `TURNFEAR`, `INCCAT`, and
> `frequency` columns, and that the frequencies sum to ≈ 1.0 **within
> each** fear group (so the two distributions are independently
> normalized).

In [ ]:
# Verify grouped DataFrame structure
assert "TURNFEAR" in df_inccat.columns, "df_inccat should have TURNFEAR column"
assert "INCCAT" in df_inccat.columns, "df_inccat should have INCCAT column"
assert "frequency" in df_inccat.columns, "df_inccat should have frequency column"
# Check frequencies sum to ~1.0 for each group
for grp in [0, 1]:
    grp_sum = df_inccat[df_inccat["TURNFEAR"] == grp]["frequency"].sum()
    assert abs(grp_sum - 1.0) < 0.01, (
        f"Frequencies for TURNFEAR={grp} should sum to ~1.0, got {grp_sum}"
    )
print("All checks passed!")

> ✅ Each group's income distribution is a valid, comparable set of
> proportions.

➡️ Single comparisons are useful, but clustering cares about how features
**relate**. Next we measure those relationships with correlation.

## 7. Analyze Correlations

### Problem

Understanding relationships between financial variables helps identify
which features might be useful for consumer segmentation. We want to
compare correlation structures between the full dataset and the
credit-fearful subset.

### Approach

We will compute one pairwise correlation first, then build full
correlation matrices for a set of key financial and demographic
variables. Comparing matrices between groups reveals how relationships
differ.

### Tasks

Compute the correlation between `ASSET` and `HOUSES` for the full
dataset. This single value (between −1 and 1) indicates how strongly the
two move together.

**Code 6.1.7.1**:

In [ ]:
# Compute correlation between ASSET and HOUSES for full dataset
asset_house_corr_full = df["ASSET"].corr(df["HOUSES"])  # correlation of df["ASSET"] with df["HOUSES"]
print("SCF: Asset-Houses Correlation:", round(asset_house_corr_full, 3))

Compute the same correlation for the credit-fearful subset. Comparing it
to the full-dataset value reveals whether the relationship differs for
this subgroup.

**Code 6.1.7.2**:

In [ ]:
# Compute same correlation for credit-fearful subset
asset_house_corr_fear = df_fear["ASSET"].corr(df_fear["HOUSES"])  # correlation for df_fear
print("Credit Fearful: Asset-Houses Correlation:",
      round(asset_house_corr_fear, 3))

Now build a correlation **matrix** for five key variables in the full
dataset. The styled output uses color gradients to highlight strong
positive and negative correlations.

**Code Tas6.1.7.3**:

In [ ]:
import jinja2
# Define columns for correlation matrix
cols = ["ASSET", "HOUSES", "INCOME", "DEBT", "EDUC"]

# Compute correlation matrix for full dataset
corr_full = df[cols].corr()  # correlation matrix for df[cols]
corr_full.style.background_gradient(axis=None)

In [ ]:
# Define columns for correlation matrix
cols = ["ASSET", "HOUSES", "INCOME", "DEBT", "EDUC"]

# Compute correlation matrix for full dataset
corr_full = df[cols].corr()
corr_full.style.background_gradient(axis=None)

Create the same correlation matrix for the credit-fearful subset. Look
for differences in the correlation patterns compared to the full dataset.

**Code 6.1.7.4**:

In [ ]:
# Compute correlation matrix for credit-fearful subset
corr_fear = df_fear[cols].corr()  # correlation matrix for df_fear[cols]
corr_fear.style.background_gradient(axis=None)

> 📊 Comparing the two matrices
>
> Put the full-population matrix next to the credit-fearful one and hunt
> for cells that **change color** between them. A pair like
> `DEBT`–`HOUSES` that is mildly correlated overall but strongly
> correlated among the fearful is a signal that this subgroup's finances
> are structured differently — exactly the kind of feature relationship
> clustering can exploit.

### Checkpoint

> 🧪 What this verifies
>
> That the scalar correlations fall in the valid `[-1, 1]` range and that
> both correlation matrices have the expected `(5, 5)` shape.

In [ ]:
# Verify correlation values are in valid range [-1, 1]
assert -1 <= asset_house_corr_full <= 1, (
    f"Correlation should be in [-1, 1], got {asset_house_corr_full}"
)
assert -1 <= asset_house_corr_fear <= 1, (
    f"Correlation should be in [-1, 1], got {asset_house_corr_fear}"
)
# Verify correlation matrix shape
assert corr_full.shape == (5, 5), (
    f"Expected correlation matrix shape (5, 5), got {corr_full.shape}"
)
assert corr_fear.shape == (5, 5), (
    f"Expected correlation matrix shape (5, 5), got {corr_fear.shape}"
)
print("All checks passed!")

> ✅ The correlation structures are well-formed for both groups.

➡️ Finally, we visualize the standout relationships — education and the
debt-vs-asset/house patterns — directly.

## 8. Visualize Education and Debt Relationships

### Problem

We observed differences in the correlation matrices. Now we want to
visualize these relationships directly through comparative bar charts and
scatter plots.

### Approach

We will draw a side-by-side bar chart comparing education levels between
the two groups, then scatter plots of debt against assets and house value
for both the full dataset and the credit-fearful subset.

### Tasks

Compute education frequencies grouped by `TURNFEAR`. We **keep** the
numeric `EDUC` codes here because they represent grade levels with a
natural ordering.

**Code 6.1.8.1**:

In [ ]:
# Compute education frequencies by TURNFEAR group
# Note: We keep numeric codes for EDUC (grade levels) for ordering
df_educ = (
    df["EDUC"]
    .groupby(df["TURNFEAR"])
    .value_counts(normalize=True)
    .rename("frequency")
    .reset_index()
)
df_educ.head()

Create a side-by-side bar chart comparing education distributions; higher
education levels appear to the right on the x-axis.

**Code 6.1.8.2**:

In [ ]:
# Create side-by-side bar chart for education
sns.barplot(x="EDUC", y="frequency", hue="TURNFEAR", data=df_educ) # x="EDUC", y="frequency", hue="TURNFEAR", data=df_educ
plt.xlabel("Education Level")
plt.ylabel("Frequency (%)")
plt.title("Educational Attainment: Credit Fearful vs. Non-fearful");

Create a scatter plot of `DEBT` vs `ASSET` for the full dataset. Each
point is a household; the pattern reveals how the two variables relate.

**Code 6.1.8.3**:

In [ ]:
# Create scatter plot of DEBT vs ASSET for full dataset
df.plot.scatter(x="DEBT", y="ASSET"); # x="DEBT", y="ASSET"

Create the **same** scatter for the credit-fearful subset. Compare the
spread and concentration of points against the full-dataset plot.

**Code 6.1.8.4**:

In [ ]:
# Create scatter plot of DEBT vs ASSET for credit-fearful subset
df_fear.plot.scatter(x="DEBT", y="ASSET");  # x="DEBT", y="ASSET"

Now scatter `DEBT` vs `HOUSES` for the full dataset — visualizing how
debt relates to home value.

**Code 6.1.8.5**:

In [ ]:
# Create scatter plot of DEBT vs HOUSES for full dataset
df.plot.scatter(x="DEBT", y="HOUSES");  # x="DEBT", y="HOUSES"

Create the same `DEBT` vs `HOUSES` scatter for the credit-fearful subset.
The correlation matrix suggested this relationship differs for this
group — see if the scatter agrees.

**Code 6.1.8.6**:

In [ ]:
# Create scatter plot of DEBT vs HOUSES for credit-fearful subset
df_fear.plot.scatter(x="DEBT", y="HOUSES");  # x="DEBT", y="HOUSES"

> 📊 Reading the education bars and debt scatters
>
> - **Education:** if the non-fearful bars dominate at the higher `EDUC`
>   codes while the fearful bars dominate at the lower ones, education
>   tracks credit confidence.
> - **Scatters:** look at the **slope and tightness** of the cloud. A
>   tighter, steeper `DEBT`–`HOUSES` cloud in the credit-fearful subset
>   than in the full population is the visual counterpart of the stronger
>   correlation we measured in §7 — for these households, debt and home
>   value rise together more lockstep.

### Checkpoint

> 🧪 What this verifies
>
> That the education DataFrame has its `EDUC` and `frequency` columns and
> that every normalized frequency lies in `[0, 1]`.

In [ ]:
# Verify education DataFrame structure
assert "EDUC" in df_educ.columns, "df_educ should have EDUC column"
assert "frequency" in df_educ.columns, "df_educ should have frequency column"
assert df_educ["frequency"].min() >= 0, "Frequencies should be non-negative"
assert df_educ["frequency"].max() <= 1, "Normalized frequencies should be <= 1"
print("All checks passed!")

> ✅ The education distribution is well-formed — and our EDA is complete.

# Wrap-up

In this notebook, you accomplished the following:

-   Loaded and inspected the Survey of Consumer Finances dataset with
    over 350 columns.
-   Created subsets based on categorical conditions (`TURNFEAR`).
-   Replaced numeric codes with human-readable labels using
    dictionaries.
-   Created bar charts, histograms, and scatter plots to explore
    variable distributions.
-   Compared subgroups using normalized frequencies and side-by-side bar
    charts.
-   Computed and compared correlation matrices to identify relationship
    differences.
-   Discovered that credit-fearful households show distinct patterns:
    younger ages, lower incomes, and a stronger correlation between debt
    and house value.

> 🧠 The bigger picture
>
> EDA only **describes** the data — it told us *which* features vary and
> *where* subgroups differ, but we drew every boundary (fearful vs. not)
> by hand using a label that already existed. Real segmentation asks the
> harder question: **what natural groups exist when no one hands us the
> labels?**

➡️ Next, you will apply **K-means clustering** to segment consumers using
two financial features (`DEBT` and `HOUSES`), building intuition for how
a clustering algorithm discovers structure on its own.